In [ ]:
# ==========================================
# 1. SETUP & DEPENDENCIES
# ==========================================
!pip install -q torch transformers pillow kaggle evaluate rouge_score

import os
import csv
import shutil
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torchvision import transforms
from sklearn.model_selection import train_test_split
from transformers import (
    ViTImageProcessor,
    GPT2TokenizerFast,
    VisionEncoderDecoderModel,
    CLIPProcessor,
    CLIPModel,
    get_linear_schedule_with_warmup
)
import kagglehub
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Config Paths
PROJECT_PATH = "/content/drive/My Drive/Cricket_Commentary_Project"
MODEL_SAVE_PATH = os.path.join(PROJECT_PATH, "final_model")
DATASET_PATH = os.path.join(PROJECT_PATH, "dataset")
CSV_PATH = os.path.join(PROJECT_PATH, "captions.csv")

os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 2. DATA PREPARATION
# ==========================================
# Download Dataset
if not os.path.exists(DATASET_PATH):
    print("⬇️ Downloading dataset...")
    path = kagglehub.dataset_download("aneesh10/cricket-shot-dataset")
    shutil.copytree(path, DATASET_PATH)

# Helper: CLIP Classifier for Drives
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def classify_shot(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        inputs = clip_processor(text=["cover drive", "straight drive"], images=image, return_tensors="pt", padding=True).to(device)
        return inputs(pixel_values=inputs.pixel_values).logits_softmax(dim=1).argmax().item()
    except: return 0

# Generate Captions CSV
if not os.path.exists(CSV_PATH):
    print("📝 Generating Labels...")
    data = []
    for root, _, files in os.walk(DATASET_PATH):
        for file in files:
            if not file.lower().endswith(('jpg', 'png')): continue
            path = os.path.join(root, file)
            folder = os.path.basename(root).lower()

            caption = "A cricket shot."
            if "drive" in folder:
                caption = "A magnificent straight drive." if classify_shot(path) == 1 else "A glorious cover drive through the gap."
            elif "pull" in folder: caption = "He pulls it away for a massive six!"
            elif "sweep" in folder: caption = "A well-executed sweep shot."
            elif "leg" in folder: caption = "He glances it off his pads to fine leg."

            data.append([path, caption])

    pd.DataFrame(data, columns=['image_path', 'caption']).to_csv(CSV_PATH, index=False)

# ==========================================
# 3. TRAINING
# ==========================================
# Load Processors
feature_extractor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Dataset Class
class CricketDataset(Dataset):
    def __init__(self, df, processor, tokenizer):
        self.df = df
        self.processor = processor
        self.tokenizer = tokenizer

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try: image = Image.open(row['image_path']).convert("RGB")
        except: image = Image.new('RGB', (224, 224), color='black')

        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze()
        labels = self.tokenizer(
            row['caption'], padding="max_length", max_length=45, truncation=True, return_tensors="pt"
        ).input_ids.squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

# Load Data
df = pd.read_csv(CSV_PATH)
train_df, val_df = train_test_split(df, test_size=0.1)
train_loader = DataLoader(CricketDataset(train_df, feature_extractor, tokenizer), batch_size=32, shuffle=True)

# Model Init
model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
    "google/vit-base-patch16-224-in21k", "gpt2"
).to(device)
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# Training Loop
optimizer = AdamW(model.parameters(), lr=2e-5)
print("🚀 Training Started...")

for epoch in range(5): # Adjust epochs as needed
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch in loop:
        pix = batch["pixel_values"].to(device)
        lbl = batch["labels"].to(device)
        loss = model(pixel_values=pix, labels=lbl).loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

# ==========================================
# 4. SAVE MODEL TO DRIVE
# ==========================================
print(f"💾 Saving model to {MODEL_SAVE_PATH}...")
model.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)
feature_extractor.save_pretrained(MODEL_SAVE_PATH)
print("✅ Model Saved Successfully! You can now use the Inference Notebook.")